In [ ]:
import numpy as np

class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.1):
        self.lr = learning_rate
        
        # Initialize weights and biases randomly
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))
        self.W3 = np.random.randn(input_size, hidden_size) * 0.01
        self.b3 = np.zeros((1, output_size))m
        
    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))
    
    def _sigmoid_derivative(self, x):
        return x * (1 - x)
    
    def forward(self, X):
        # Forward propagation
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = self._sigmoid(self.Z1)
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = self._sigmoid(self.Z2)
        return self.A2
        
    def backward(self, X, y, output):
        # Backpropagation: Calculate error gradients
        output_error = y - output
        output_delta = output_error * self._sigmoid_derivative(output)
        
        hidden_error = np.dot(output_delta, self.W2.T)
        hidden_delta = hidden_error * self._sigmoid_derivative(self.A1)
        
        # Update weights and biases (Gradient Descent)
        self.W2 += np.dot(self.A1.T, output_delta) * self.lr
        self.b2 += np.sum(output_delta, axis=0, keepdims=True) * self.lr
        self.W1 += np.dot(X.T, hidden_delta) * self.lr
        self.b1 += np.sum(hidden_delta, axis=0, keepdims=True) * self.lr

    def train(self, X, y, epochs):
        for epoch in range(epochs):
            output = self.forward(X)
            self.backward(X, y, output)
            if epoch % 1000 == 0:
                loss = np.mean(np.square(y - output)) # Mean Squared Error
                print(f"Epoch {epoch} - Loss: {loss:.4f}")

# --- Example Usage (Solving the XOR Problem) ---
if __name__ == "__main__":
    # Inputs (4 samples, 2 features)
    X = np.array([[0,0], [0,1], [1,0], [1,1]])
    # Target outputs
    y = np.array([[0], [1], [1], [0]])
    
    # Create network: 2 inputs, 4 hidden nodes, 1 output node
    nn = NeuralNetwork(input_size=2, hidden_size=4, output_size=1, learning_rate=0.5)
    
    print("Training the network...")
    nn.train(X, y, epochs=5000)
    
    print("\nPredictions after training:")
    predictions = nn.forward(X)
    for sample, pred in zip(X, predictions):
        print(f"Input: {sample} -> Predicted: {pred[0]:.4f} (Target: {sample[0]^sample[1]})")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define the network architecture
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.hidden = nn.Linear(2, 4)  # 2 inputs to 4 hidden nodes
        self.sigmoid = nn.Sigmoid()
        self.output = nn.Linear(4, 1)  # 4 hidden nodes to 1 output
        
    def forward(self, x):
        x = self.sigmoid(self.hidden(x))
        x = self.sigmoid(self.output(x))
        return x

# Dataset (XOR problem)
X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

# Initialize network, optimizer, and loss tracking
model = SimpleNet()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.5)

# Training loop
for epoch in range(5001):
    optimizer.zero_grad()          # Clear past gradients
    predictions = model(X)         # Forward pass
    loss = criterion(predictions, y) # Compute loss
    loss.backward()                # Backpropagation
    optimizer.step()               # Update weights
    
    if epoch % 1000 == 0:
        print(f"Epoch {epoch} - Loss: {loss.item():.4f}")
